# Stage 6 - Retrospective nested Latent Player Archetype Shrinkage

**Notebook 58 - outer fold 2, release R2**

This notebook searches the exact frozen 256-candidate registry for one layer and one whole-game outer fold. Candidate, model-family, hyperparameter, and iteration selection use only inner train/validation rows; scalar temperature uses only calibration; the outer test is scored once after all choices are fixed. v6e-1 TPU is preferred for its host CPU capacity, but fitting remains host-CPU work and CPU/GPU/TPU sessions are accepted. Action timing is excluded.


## Operator settings


In [ ]:
REQUESTED_WORKERS = "auto"  # @param {type:"string"}
NOTEBOOK_ACCELERATOR = "TPU"  # @param ["CPU", "GPU", "TPU"]
LAYER = "latent_player_archetype_shrinkage"
FOLD_INDEX = 2
import os
REQUESTED_WORKERS = os.environ.get("STAGE6_REQUESTED_WORKERS", REQUESTED_WORKERS).strip()
NOTEBOOK_ACCELERATOR = NOTEBOOK_ACCELERATOR.strip().upper()
print({
    "layer": LAYER,
    "fold_index": FOLD_INDEX,
    "requested_workers": REQUESTED_WORKERS,
    "accepted_runtime_types": ["CPU", "GPU", "TPU"],
    "preferred_notebook_host": "v6e-1 TPU",
    "logical_data_lanes": 48,
    "candidate_registry_size": 256,
    "candidate_evaluations": 384,
    "candidate_internal_threads": 1,
    "model_execution_device": "host_cpu_process",
    "tree_training_uses_gpu_or_tpu_acceleration": False,
})


## Mount Drive and verify the three exact packages


In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

for variable in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[variable] = "1"

PINNED = {"catboost": "1.2.10", "joblib": "1.5.2", "lightgbm": "4.6.0", "numpy": "2.2.6", "pandas": "2.3.3", "psutil": "7.0.0", "pyarrow": "21.0.0", "scikit-learn": "1.7.2", "threadpoolctl": "3.6.0", "xgboost": "3.3.0"}
MODULES = {"catboost": "catboost", "joblib": "joblib", "lightgbm": "lightgbm", "numpy": "numpy", "pandas": "pandas", "psutil": "psutil", "pyarrow": "pyarrow", "scikit-learn": "sklearn", "threadpoolctl": "threadpoolctl", "xgboost": "xgboost"}
installed = {name: (importlib.metadata.version(name) if importlib.util.find_spec(MODULES[name]) else None) for name in PINNED}
mismatched = {name: version for name, version in PINNED.items() if installed[name] != version}
preloaded = [MODULES[name] for name in mismatched if MODULES[name] in sys.modules]
if mismatched:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *[f"{name}=={version}" for name, version in PINNED.items()]])
verified = {name: importlib.metadata.version(name) for name in PINNED}
if verified != PINNED:
    raise RuntimeError(f"PINNED_ENVIRONMENT_INSTALL_FAILED: {verified}")
if preloaded:
    raise RuntimeError("PINNED_ENVIRONMENT_RESTART_REQUIRED: restart this session and Run all; " + ",".join(preloaded))

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("This release notebook is intended for Google Colab") from error
drive.mount("/content/drive")

STAGE6_DRIVE_ROOT = Path("/content/drive/MyDrive/Poker Assistant Model/Stage 6 - Qualified Layers Improvement")
packages = [
    (STAGE6_DRIVE_ROOT / "inputs" / "POKERNOW_ASSISTANT_STAGE6_BEHAVIOR_CHALLENGER_V2_SCIENCE_R14.zip", 386673, "624f8a4623307b49eee4a0e2331df33485709c94a3aad2c5a731ee894a633971", Path("/content/stage6-nested-base-r14")),
    (STAGE6_DRIVE_ROOT / "inputs" / "POKERNOW_ASSISTANT_STAGE6_ADDITIONAL_EXPLOITATION_RELEASE_R21.zip", 79218, "c4e74aa7842d58fdd5b127be9dffff0c5b021a1b843ca7ef92cebd3dcf9c4888", Path("/content/stage6-nested-additional-r21")),
    (STAGE6_DRIVE_ROOT / "inputs" / "POKERNOW_ASSISTANT_STAGE6_RETROSPECTIVE_NESTED_SEARCH_R2.zip", 26502, "2c118ff3fb9dab7401851e87c2e793b32c9dd1086be2c50055cded0739fc933d", Path("/content/stage6-nested-search-overlay-r2")),
]
for package_path, expected_bytes, expected_sha256, package_root in packages:
    if not package_path.is_file() or package_path.stat().st_size != expected_bytes:
        raise RuntimeError(f"package size mismatch: {package_path.name}")
    digest = hashlib.sha256()
    with package_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected_sha256:
        raise RuntimeError(f"package SHA-256 mismatch: {package_path.name}")
    if package_root.exists():
        shutil.rmtree(package_root)
    package_root.mkdir(parents=True)
    with zipfile.ZipFile(package_path) as archive:
        names = archive.namelist()
        if not names or names[-1] != "PACKAGE_MANIFEST.json" or len(names) != len(set(names)):
            raise RuntimeError(f"invalid package inventory: {package_path.name}")
        for name in names:
            pure = PurePosixPath(name)
            if pure.is_absolute() or ".." in pure.parts or "\\" in name:
                raise RuntimeError(f"unsafe package member: {name}")
        archive.extractall(package_root)
    manifest = json.loads((package_root / "PACKAGE_MANIFEST.json").read_text(encoding="utf-8"))
    expected_self = manifest.pop("self_sha256")
    observed_self = hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode()).hexdigest()
    manifest["self_sha256"] = expected_self
    if observed_self != expected_self or manifest["member_order"] != names[:-1]:
        raise RuntimeError(f"package manifest mismatch: {package_path.name}")
    for member in manifest["members"]:
        raw = (package_root / member["path"]).read_bytes()
        if len(raw) != member["bytes"] or hashlib.sha256(raw).hexdigest() != member["sha256"]:
            raise RuntimeError(f"package member mismatch: {member['path']}")

for source_root in [packages[0][3] / "src", packages[1][3] / "src", packages[2][3] / "src"]:
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))
print(json.dumps({
    "event": "stage6_retrospective_nested_search_release_ready",
    "packages": [row[0].name for row in packages],
    "logical_data_lanes": 48,
    "candidate_internal_threads": 1,
    "model_execution_device": "host_cpu_process",
    "tree_training_uses_gpu_or_tpu_acceleration": False,
    "authority_class": "retrospective_nested_out_of_fold",
    "fresh_or_prospective_claim": False,
}, sort_keys=True))


## Run the exact nested layer/fold search


In [ ]:
from stage6_retrospective_nested_search import run_retrospective_nested_layer_fold_search
RESULT = run_retrospective_nested_layer_fold_search(
    STAGE6_DRIVE_ROOT,
    LAYER,
    FOLD_INDEX,
    requested_workers=REQUESTED_WORKERS,
    accelerator=NOTEBOOK_ACCELERATOR,
)
print(json.dumps(RESULT, sort_keys=True, indent=2))


## Release memory and download the compact returned result


In [ ]:
import gc
gc.collect()
try:
    from google.colab import files
    files.download(RESULT["returned_result"]["path"])
except Exception as error:
    print({"download_status": "manual_download_may_be_required", "detail": str(error)})
